# Eksperimen Hands-on: Evolusi Payload Input ke LLM

Notebook ini dirancang khusus untuk mempraktikkan **bagaimana bentuk payload yang kita kirimkan ke LLM berevolusi**, mulai dari:
1. **Tahap 1:** Satu teks mentah tunggal (*single-turn raw string*)
2. **Tahap 2:** Penggabungan teks manual (*string template & concat/replace*)
3. **Tahap 3:** Daftar pesan terstruktur dengan peran (*list of messages with roles*)
4. **Tahap 4:** Injeksi data eksternal dinamis (*RAG payload*)
5. **Tahap 5:** Percakapan bersambung dengan riwayat pesan (*multi-turn conversation loop*)

Jalankan sel dari atas ke bawah untuk melihat bagaimana perubahan payload mempengaruhi respons model.

## 0. Setup Environment & Inisialisasi Client

In [3]:
import os
from getpass import getpass
from openai import OpenAI, OpenAIError
from IPython.display import Markdown, display

# Ambil API Key dari environment jika ada, atau minta input jika belum ada
api_key = os.environ.get("OPENAI_API_KEY", "").strip()
if not api_key:
    api_key = getpass("Masukkan OpenAI API Key: ").strip()

client = OpenAI(api_key=api_key)
MODEL_NAME = os.environ.get("AGENT_MODEL", "gpt-6-luna")
print(f"Client siap menggunakan model: {MODEL_NAME}")

Client siap menggunakan model: gpt-6-luna


---
## Tahap 1: String Tunggal (Versi Awal)

Di sini kita hanya mengirimkan satu teks biasa. Tidak ada pemisahan peran sistem, tidak ada riwayat, dan tidak ada konteks tambahan.

In [ ]:
# Payload Tahap 1: HANYA SATU STRING BIASA
prompt_tahap_1 = "Halo, saya Bima. bagaimana cara membuat file python di development bisa baca environment variables?"

print("=== BENTUK PAYLOAD YANG DIKIRIM ===")
print(type(prompt_tahap_1))
print(prompt_tahap_1)
print("====================================\n")

response_1 = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[{"role": "user", "content": prompt_tahap_1}],
    max_completion_tokens=800
)

display(Markdown(f"**Jawaban LLM:**\n{response_1.choices[0].message.content}"))

=== BENTUK PAYLOAD YANG DIKIRIM ===
<class 'str'>
Halo, saya Bima. bagaimana cara membuat file python di development bisa baca environment variables?



**Jawaban LLM:**
Halo Bima! Di Python, environment variables bisa dibaca dengan modul bawaan `os`.

```python
import os

api_key = os.getenv("API_KEY")  # None jika belum disetel
print(api_key)
```

Atau gunakan nilai bawaan:

```python
api_key = os.getenv("API_KEY", "nilai-default")
```

### Menyetel variabel sebelum menjalankan program

**macOS/Linux:**

```bash
API_KEY="contoh-kunci" python app.py
```

**Windows PowerShell:**

```powershell
$env:API_KEY = "contoh-kunci"
python app.py
```

### Memakai file `.env` saat development

Buat file `.env` di folder proyek:

```env
API_KEY=contoh-kunci
```

Pasang paketnya:

```bash
pip install python-dotenv
```

Lalu muat file tersebut di Python:

```python
import os
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("API_KEY")
```

Tambahkan `.env` ke `.gitignore` agar tidak ikut terunggah ke Git—terutama jika berisi password atau API key. Variabel yang baru disetel biasanya baru terbaca setelah proses Python dijalankan ulang.

---
## Tahap 2: Template String & Concat / Replace Manual

Di tahap ini, instruksi mulai dibuat lebih rapi dengan blok `[ROLE]`, `[TASK]`, dan `<data>`, tetapi kita masih menggabungkannya secara manual menjadi satu teks string panjang sebelum dikirim.

In [ ]:
template_prompt = """
[ROLE]
Anda adalah Asisten HR Nusantara Digital. Jawab singkat dan profesional.

[CONTEXT]
Nama Karyawan: <<NAMA>>
Status: Karyawan Tetap

[TASK]
Jawab pertanyaan berikut:
<<PERTANYAAN>>

[FORMAT]
Sapa karyawan dengan namanya, lalu berikan jawaban dalam maksimal 2 kalimat.
"""

# Penggabungan manual menggunakan string replace / f-string
prompt_tahap_2 = template_prompt.replace("<<NAMA>>", "Bima").replace(
    "<<PERTANYAAN>>", "Apakah saya bisa mengajukan cuti 5 hari di bulan pertama bekerja?"
)

print("=== BENTUK PAYLOAD YANG DIKIRIM (SATU STRING HASIL CONCAT) ===")
print(prompt_tahap_2)
print("==============================================================\n")

response_2 = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[{"role": "user", "content": prompt_tahap_2}],
    max_completion_tokens=800
)

display(Markdown(f"**Jawaban LLM:**\n{response_2.choices[0].message.content}"))

---
## Tahap 3: Pemisahan Peran Terstruktur (`list of dict`)

**Titik Balik:** Alih-alih menggabungkan instruksi sistem dan ucapan pengguna menjadi satu string, API modern menerima daftar objek berlabel peran (`system`, `user`, `assistant`).

* `system`: Aturan main, gaya bahasa, persona, batasan keamanan.
* `user`: Teks yang diketik oleh pengguna.

In [ ]:
# Payload Tahap 3: LIST OF DICTIONARIES
payload_tahap_3 = [
    {
        "role": "system",
        "content": "Anda adalah Asisten Kebijakan HR. Jawab dengan gaya formal dan selalu tolak permintaan yang melanggar privasi."
    },
    {
        "role": "user",
        "content": "Berapa saldo cuti rekan kerja saya yang bernama Alya?"
    }
]

print("=== BENTUK PAYLOAD YANG DIKIRIM (ARRAY BERDASARKAN PERAN) ===")
import json
print(json.dumps(payload_tahap_3, indent=2))
print("==============================================================\n")

response_3 = client.chat.completions.create(
    model=MODEL_NAME,
    messages=payload_tahap_3,
    max_tokens=200
)

display(Markdown(f"**Jawaban LLM:**\n{response_3.choices[0].message.content}"))

---
## Tahap 4: Payload Dinamis RAG (Injeksi Konteks Dokumen)

Pada RAG (Retrieval-Augmented Generation), payload input diperkaya dengan hasil pencarian database dokumen kebijakan sebelum diteruskan ke LLM.

In [ ]:
# Simulasi dokumen yang ditemukan dari database vektor/PostgreSQL
mock_retrieved_chunk = """
[DOKUMEN: HR-LEAVE-001]
Judul: Kebijakan Cuti Tahunan Nusantara Digital
Pasal 4: Karyawan tetap berhak atas 12 hari kerja cuti tahunan setelah melewati masa probation (3 bulan).
Pasal 5: Pengambilan cuti lebih dari 3 hari berturut-turut wajib diajukan minimal 7 hari sebelumnya.
"""

# Payload RAG menyuntikkan dokumen tersebut ke dalam instruksi atau konteks
payload_tahap_4 = [
    {
        "role": "system",
        "content": (
            "Anda adalah asisten HR. Jawab HANYA berdasarkan dokumen yang disediakan di bawah. "
            "Sertakan sitasi kode dokumen [HR-LEAVE-xxx]. Jika tidak ada di dokumen, katakan tidak tahu.\n\n"
            f"=== DOKUMEN KEBIJAKAN ===\n{mock_retrieved_chunk}\n========================"
        )
    },
    {
        "role": "user",
        "content": "Berapa hari sebelum pelaksanaan saya harus mengajukan cuti 4 hari berturut-turut?"
    }
]

print("=== BENTUK PAYLOAD TAHAP 4 (RAG GROUNDED) ===")
print(f"Jumlah Pesan: {len(payload_tahap_4)}")
print(f"Ukuran Karakter Konteks Dokumen: {len(mock_retrieved_chunk)} chars")
print("============================================\n")

response_4 = client.chat.completions.create(
    model=MODEL_NAME,
    messages=payload_tahap_4,
    max_tokens=250
)

display(Markdown(f"**Jawaban LLM:**\n{response_4.choices[0].message.content}"))

---
## Tahap 5: Percakapan Berkelanjutan (*Multi-Turn Conversation Loop*)

Sekarang kita buktikan mengapa chatbot terasa "ingat":
1. **Percakapan 1:** User memperkenalkan nama dan bertanya saldo.
2. **Balasan 1:** Asisten menjawab dan disimpan ke list.
3. **Percakapan 2:** User bertanya *"Siapa nama saya dan berapa saldo saya?"* tanpa menyebutkan namanya lagi.

Mari kita coba **Dua Skenario**: Tanpa Riwayat vs Dengan Riwayat.

In [ ]:
print("=== SKENARIO A: TANPA RIWAYAT (HANYA MENGIRIM PERTANYAAN BARU) ===")
# Panggilan 1
turn_1_messages = [{"role": "user", "content": "Halo, nama saya Bima Pratama. Saya karyawan baru."}]
res_a1 = client.chat.completions.create(model=MODEL_NAME, messages=turn_1_messages, max_tokens=100)
print("Turn 1 Answer:", res_a1.choices[0].message.content)

# Panggilan 2: Lupa mengirimkan Turn 1
turn_2_messages_tanpa_riwayat = [{"role": "user", "content": "Siapa nama lengkap saya?"}]
res_a2 = client.chat.completions.create(model=MODEL_NAME, messages=turn_2_messages_tanpa_riwayat, max_tokens=100)
print("Turn 2 Answer (Tanpa Riwayat):", res_a2.choices[0].message.content)
print("\n-> Bukti: Model LUPA karena Turn 1 tidak dikirimkan ulang!")

In [ ]:
print("=== SKENARIO B: DENGAN RIWAYAT (MENYIMPAN & MENGIRIM ULANG SEMUA PESAN) ===\n")

# Inisialisasi wadah riwayat obrolan (di backend ini disimpan di SQLite / PostgreSQL)
chat_history = [
    {"role": "system", "content": "Anda adalah asisten HR yang ramah dan mengingat nama lawan bicara."}
]

# Fungsi pembantu untuk mengirim pesan dan otomatis memperbarui riwayat
def kirim_pesan_dengan_riwayat(pesan_baru: str):
    # 1. Tambahkan pesan user ke riwayat
    chat_history.append({"role": "user", "content": pesan_baru})
    
    # 2. Kirim SELURUH riwayat ke LLM
    res = client.chat.completions.create(
        model=MODEL_NAME,
        messages=chat_history,
        max_tokens=200
    )
    jawaban = res.choices[0].message.content
    
    # 3. Tambahkan balasan asisten ke riwayat agar diingat pada putaran berikutnya
    chat_history.append({"role": "assistant", "content": jawaban})
    
    return jawaban

# Jalankan Turn 1
jawaban_1 = kirim_pesan_dengan_riwayat("Halo, nama saya Bima Pratama. Saya karyawan baru di divisi IT.")
print("[User]: Halo, nama saya Bima Pratama. Saya karyawan baru di divisi IT.")
print(f"[Bot]: {jawaban_1}\n")

# Jalankan Turn 2
jawaban_2 = kirim_pesan_dengan_riwayat("Siapa nama lengkap dan divisi saya?")
print("[User]: Siapa nama lengkap dan divisi saya?")
print(f"[Bot]: {jawaban_2}\n")

print("=== BUKTI ISI RIWAYAT YANG TERSIMPAN DI MEMORI KITA ===")
for idx, m in enumerate(chat_history):
    print(f"{idx}. [{m['role']}]: {m['content'][:60]}...")

---
## 6. Ringkasan Pelajaran

1. **LLM tidak pernah mengingat:** Yang menyimpan data adalah aplikasi Python Anda (`chat_history`).
2. **Payload tidak di-concat manual:** API LLM menerima struktur `list[dict]` berisi pasangan `{"role": ..., "content": ...}`.
3. **Skalabilitas:** Jika percakapan sudah sangat panjang (misal 50 pesan), kita bisa membatasi riwayat dengan teknik *sliding window* (misalnya hanya mengirimkan 10 pesan terakhir: `chat_history[-10:]`) agar tidak melebihi kapasitas context window dan menghemat biaya token.